# Project Intelligence Notebook

Reads all documents from the `project_documents` Dataiku managed folder, processes each one
through a Claude-powered agent that builds a persistent memory of the document corpus, then
synthesizes everything into a comprehensive project intelligence report.

**Supported formats:** PDF, Word (.docx), PowerPoint (.pptx), Excel (.xlsx)

**Output:** A detailed report covering project history, decisions, architecture, timeline,
team, challenges, and cross-document insights.

---
**Prerequisites:**
- `ANTHROPIC_API_KEY` set in the Dataiku API keys settings or as an environment variable
- A Dataiku managed folder named `project_documents` containing the project files

In [ ]:
import os
import sys
import dataiku
import anthropic
from tqdm import tqdm

# Add src to path when running inside Dataiku
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from src.document_parser import parse_document
from src.memory import MemoryStore
from src.agent import analyze_document
from src.report_generator import extract_global_insights, generate_report

## Configuration

In [ ]:
# --- Configuration ---
FOLDER_NAME = "project_documents"       # Dataiku managed folder name
PROJECT_NAME = "My Technology Project"  # Human-readable name for the report
MEMORY_PATH = "agent_memory.json"       # Where to persist the agent's memory
REPORT_OUTPUT_PATH = "project_intel_report.md"  # Where to save the final report
SKIP_ALREADY_PROCESSED = True           # Resume from where we left off

# Supported file extensions
SUPPORTED_EXTENSIONS = {".pdf", ".docx", ".doc", ".pptx", ".ppt", ".xlsx", ".xls"}

# Initialize the Anthropic client
api_key = os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    # Try Dataiku variable
    try:
        api_key = dataiku.get_custom_variables().get("ANTHROPIC_API_KEY")
    except Exception:
        pass

if not api_key:
    raise ValueError(
        "ANTHROPIC_API_KEY not found. Set it as an environment variable "
        "or in Dataiku project variables."
    )

client = anthropic.Anthropic(api_key=api_key)
print("Anthropic client initialized.")

## Step 1 — Load documents from the Dataiku folder

In [ ]:
folder = dataiku.Folder(FOLDER_NAME)

all_paths = folder.list_paths_in_partition()
document_paths = [
    p for p in all_paths
    if os.path.splitext(p.lower())[1] in SUPPORTED_EXTENSIONS
]

print(f"Found {len(all_paths)} total files in '{FOLDER_NAME}'.")
print(f"Found {len(document_paths)} supported documents to process.")

# Show breakdown by type
from collections import Counter
ext_counts = Counter(os.path.splitext(p.lower())[1] for p in document_paths)
for ext, count in sorted(ext_counts.items()):
    print(f"  {ext}: {count} files")

## Step 2 — Initialize memory store

In [ ]:
memory = MemoryStore(memory_path=MEMORY_PATH)
memory.memory.project_name = PROJECT_NAME
memory.save()

already_done = [p for p in document_paths if memory.is_processed(os.path.basename(p))]
to_process = (
    [p for p in document_paths if not memory.is_processed(os.path.basename(p))]
    if SKIP_ALREADY_PROCESSED
    else document_paths
)

print(f"Already processed: {len(already_done)} documents")
print(f"Remaining to process: {len(to_process)} documents")
stats = memory.get_stats()
print(f"Memory stats: {stats}")

## Step 3 — Process documents through the agent

In [ ]:
errors = []

for path in tqdm(to_process, desc="Analyzing documents"):
    filename = os.path.basename(path)
    print(f"\n→ Processing: {filename}")

    try:
        # Read file bytes from Dataiku folder
        with folder.get_download_stream(path) as stream:
            file_bytes = stream.read()

        # Parse document content
        parsed = parse_document(filename, file_bytes)

        if not parsed.content.strip():
            print(f"  ⚠ No content extracted from {filename}, skipping.")
            continue

        content_length = len(parsed.content)
        print(f"  Extracted {content_length:,} characters")

        # Analyze with Claude agent
        record = analyze_document(client, parsed, memory)
        print(f"  ✓ Summary: {record.summary[:120]}...")
        print(f"  Topics: {', '.join(record.key_topics[:5])}")

    except Exception as e:
        print(f"  ✗ Error processing {filename}: {e}")
        errors.append({"file": filename, "error": str(e)})

print(f"\n✓ Processing complete. {len(to_process) - len(errors)} succeeded, {len(errors)} errors.")
if errors:
    print("Errors:", errors)

## Step 4 — Extract cross-document insights

In [ ]:
print("Extracting cross-document patterns and timeline...")
insights = extract_global_insights(client, memory)

print(f"\nGlobal themes identified: {len(insights.get('global_themes', []))}")
for theme in insights.get('global_themes', []):
    print(f"  • {theme}")

print(f"\nTimeline events: {len(insights.get('project_timeline', []))}")
print(f"Key decisions: {len(insights.get('key_decisions', []))}")
print(f"Cross-document insights: {len(insights.get('cross_document_insights', []))}")

## Step 5 — Generate the final project intelligence report

In [ ]:
report = generate_report(client, memory, project_name=PROJECT_NAME)

# Save report to file
with open(REPORT_OUTPUT_PATH, "w") as f:
    f.write(report)

print(f"\n\n✓ Report saved to: {REPORT_OUTPUT_PATH}")
print(f"Report length: {len(report):,} characters")

## Step 6 — (Optional) Save report to a Dataiku dataset

In [ ]:
# Optionally write the report to a Dataiku managed folder for sharing
# Uncomment and set the output folder name

# OUTPUT_FOLDER_NAME = "project_intel_output"
# output_folder = dataiku.Folder(OUTPUT_FOLDER_NAME)
# with output_folder.get_writer("project_intel_report.md") as writer:
#     writer.write(report.encode("utf-8"))
# print(f"Report written to Dataiku folder: {OUTPUT_FOLDER_NAME}")

## Step 7 — Display report preview

In [ ]:
from IPython.display import Markdown, display

# Display the first 5000 characters as a preview
preview = report[:5000] + ("\n\n*[Report truncated for preview — see full file]*" if len(report) > 5000 else "")
display(Markdown(preview))

## Memory Summary

In [ ]:
import json

final_stats = memory.get_stats()
print("=== Agent Memory Summary ===")
print(f"Project: {memory.memory.project_name}")
print(f"Documents processed: {final_stats['total_documents']}")
print(f"File types: {', '.join(final_stats['file_types'])}")
print(f"Technologies identified: {final_stats['total_technologies']}")
print(f"Technologies: {', '.join(memory.memory.technologies_used[:20])}")
print(f"Stakeholders: {', '.join(memory.memory.stakeholders[:20])}")
print(f"Memory file: {MEMORY_PATH}")